# 07 — Select useful pair-TE features

120個のNested Pair Target Encodingを全モデルへ一律投入せず、役に立つ候補だけを残します。
選択は次の二段階です。

1. leakage-safeなOOF TE自身の単変量AUCで候補を順位付けし、ほぼ同じ情報を持つ列を相関で間引く
2. top-k集合を各モデルへ追加し、baseとの差を同じfoldで比較する

XGBoost、Logistic Regression、MLPは特徴量の使い方が異なるため、採用する列数と列集合を
モデルごとに保存します。改善が不安定なら0列、つまりbaseへ戻すことも正式な選択結果です。

In [ ]:
from functools import partial
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import (
    build_nested_pair_te_for_outer_fold,
    get_base_features,
    get_categorical_pairs,
    load_nested_pair_te,
)
from hpo import HPO_MODELS, save_json
from selection import (
    candidate_feature_counts,
    load_oof_te_matrix,
    prune_correlated_features,
    rank_te_features,
    select_conservative_candidate,
)
from train import make_model, run_cv
from validation import add_stratified_folds

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN
SELECTION_SEED = 2025

train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0})
assert train[TARGET].notna().all()

BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
PAIR_COLUMNS = get_categorical_pairs(BASE_CAT)
print("base:", len(BASE_FEATURES), "pair candidates:", len(PAIR_COLUMNS))

## 選択専用foldとNested TE

Notebook 06-FE2の比較に使ったseed=42とは別に、seed=2025の選択用foldを作ります。
TEはfold定義に依存するため、選択用にもnested encodingを作り直します。

これは外部holdoutではなく、fold配置だけを変えたtuning CVです。最終スコアは、このfoldではなく
Notebook 08で元のseed=42 foldへ選択結果とHPOパラメータを固定して確認します。

In [ ]:
selection_train = add_stratified_folds(
    train,
    TARGET,
    n_splits=Baseline.N_SPLITS,
    random_state=SELECTION_SEED,
    fold_column=FOLD_COLUMN,
)
selection_fold_path = ROOT / "output" / "selection_folds.csv"
selection_train[[ID_COLUMN, FOLD_COLUMN]].to_csv(selection_fold_path, index=False)

SELECTION_TE_DIR = ROOT / "output" / "nested_pair_te_selection"
SELECTION_TE_DIR.mkdir(parents=True, exist_ok=True)
REBUILD_SELECTION_TE = False

for outer_fold in sorted(selection_train[FOLD_COLUMN].unique()):
    fold_tag = str(outer_fold).replace("/", "_")
    paths = {
        "train": SELECTION_TE_DIR / f"outer_fold_{fold_tag}_train.parquet",
        "valid": SELECTION_TE_DIR / f"outer_fold_{fold_tag}_valid.parquet",
        "test": SELECTION_TE_DIR / f"outer_fold_{fold_tag}_test.parquet",
    }
    if not REBUILD_SELECTION_TE and all(path.exists() for path in paths.values()):
        print("reuse selection TE:", outer_fold)
        continue

    train_te, valid_te, test_te = build_nested_pair_te_for_outer_fold(
        train=selection_train,
        test=test,
        target=TARGET,
        categorical_pairs=PAIR_COLUMNS,
        outer_fold=outer_fold,
        id_column=ID_COLUMN,
        fold_column=FOLD_COLUMN,
        n_inner_splits=Baseline.N_SPLITS,
        smoothing=20.0,
        random_state=SELECTION_SEED,
    )
    train_te.to_parquet(paths["train"], index=False)
    valid_te.to_parquet(paths["valid"], index=False)
    test_te.to_parquet(paths["test"], index=False)
    print("saved selection TE:", outer_fold)

## Stage 1 — 安価なscreening

各outer-validationファイルをIDで結合し、全行分のleakage-safe OOF TE行列を作ります。
単変量AUCは最終的な採用基準ではなく、120候補の探索順を決めるscreeningです。

次に、AUCが高い順に候補を見て、絶対相関0.995以上のほぼ重複した列を落とします。
相関は計算量を抑えるため固定seedの最大100,000行で推定します。

In [ ]:
oof_te = load_oof_te_matrix(
    selection_train,
    SELECTION_TE_DIR,
    id_column=ID_COLUMN,
    fold_column=FOLD_COLUMN,
)
ranking = rank_te_features(oof_te, selection_train[TARGET])
ranked_after_pruning, screening_audit = prune_correlated_features(
    oof_te,
    ranking,
    threshold=0.995,
    sample_size=100_000,
    random_state=SELECTION_SEED,
)
screening_path = ROOT / "artifacts" / "te_feature_screening.csv"
screening_audit.to_csv(screening_path, index=False)

print("before pruning:", oof_te.shape[1])
print("after pruning :", len(ranked_after_pruning))
display(screening_audit.head(30))

## Stage 2 — モデル別top-k ablation

`{0, 5, 10, 20, 40, 80, all}`の候補数を比較します。0列は必ず含め、追加特徴量なしのbaseを
同じselection foldで測り直します。

採用判断には単純な最大AUCではなく、各foldでbaseとの差を取り、
`mean delta - 1 standard error`が最小改善幅を超える候補を使います。
これにより、一つのfoldだけで偶然伸びた大きな特徴量集合を選びにくくします。

In [ ]:
counts = candidate_feature_counts(len(ranked_after_pruning))
all_fold_rows = []
selection_payload = {
    "selection_seed": SELECTION_SEED,
    "screening": {
        "correlation_threshold": 0.995,
        "ranked_features_after_pruning": ranked_after_pruning,
    },
    "models": {},
}

for model_name in HPO_MODELS:
    print("=" * 70)
    print("feature selection:", model_name)
    model_fold_rows = []

    for n_features in counts:
        selected_te = ranked_after_pruning[:n_features]
        numeric_features = BASE_NUM + selected_te
        model = make_model(
            model_name,
            numeric_features,
            BASE_CAT,
            seed=Baseline.SEED,
            use_gpu=False,
        )

        if selected_te:
            feature_loader = partial(
                load_nested_pair_te,
                feature_dir=SELECTION_TE_DIR,
                selected_te_features=selected_te,
            )
        else:
            feature_loader = None

        result = run_cv(
            model=model,
            train=selection_train,
            test=test,
            features=BASE_FEATURES,
            target=TARGET,
            id_column=ID_COLUMN,
            fold_column=FOLD_COLUMN,
            label=f"selection_{model_name}_top{n_features}",
            fold_feature_loader=feature_loader,
            predict_test=False,
        )
        fold_result = result["fold_df"][["fold", "auc"]].copy()
        fold_result.insert(0, "n_te_features", n_features)
        fold_result.insert(0, "model", model_name)
        model_fold_rows.append(fold_result)
        all_fold_rows.append(fold_result)

    model_fold_scores = pd.concat(model_fold_rows, ignore_index=True)
    chosen = select_conservative_candidate(
        model_fold_scores,
        minimum_improvement=5e-5,
    )
    selected_count = int(chosen["n_te_features"])
    selected_features = ranked_after_pruning[:selected_count]
    selection_payload["models"][model_name] = {
        "selected_count": selected_count,
        "selected_features": selected_features,
        "selection_mean_auc": float(chosen["mean_auc"]),
        "mean_delta_vs_base": float(chosen["mean_delta"]),
        "conservative_delta": float(chosen["conservative_delta"]),
    }
    print("selected:", selected_count, "features")

feature_selection_folds = pd.concat(all_fold_rows, ignore_index=True)
feature_selection_folds.to_csv(
    ROOT / "artifacts" / "feature_selection_fold_scores.csv", index=False
)
save_json(
    selection_payload,
    ROOT / "artifacts" / "selected_te_features.json",
)

## 選択結果

ここで保存したJSONをNotebook 08が読み込みます。モデル間で同じ列数になる必要はありません。

In [ ]:
selected_summary = pd.DataFrame(
    [
        {
            "model": model_name,
            **{
                key: value
                for key, value in model_result.items()
                if key != "selected_features"
            },
        }
        for model_name, model_result in selection_payload["models"].items()
    ]
)
display(selected_summary)
for model_name, model_result in selection_payload["models"].items():
    print(model_name, model_result["selected_features"])

## 判断上の注意

特徴量を選ぶ行為自体がselection CVへ適合します。このNotebookの最高AUCを最終性能とは呼びません。
Notebook 08では、選択された列名を固定してHPOを行い、その後に別seedの元foldでconfirmation OOFを
作ります。改善幅が小さいときは、少ない特徴量または0列を優先します。